# PPO vs SAC Overview

## 1. PPO — "collect fresh experience, then learn from it"

```
Actor
  ↓
Environment
  ↓
(s, a, r, s')
  ↓
Rollout
  ↓
calculate advantage
  ↓
train Actor + Critic
  ↓
discard rollout
  ↓
collect fresh experience
```

PPO is on-policy.

It basically says:

"I want to learn from what my current policy just did."

It can reuse that rollout for several epochs, but after that the data becomes stale and is discarded.

### PPO's major mechanism

It prevents the actor from changing too aggressively:

```
old policy → new policy
             ↑
        clipped update
```

So PPO's superpower is:

**stable policy updates**.

---

## 2. SAC — "keep experience and learn from it repeatedly"

```
Actor
  ↓
Environment
  ↓
(s, a, r, s')
  ↓
Replay Buffer
  ↓
sample random batch
  ↓
Critics learn
  ↓
Actor learns
  ↓
KEEP buffer
  ↓
collect more experience
  ↓
sample old + new experience
```

SAC is off-policy.

It says:

"I don't care whether this experience was generated by my current policy. If it's useful, keep it and learn from it."

That's where the replay buffer matters.

---

## 3. The biggest difference

| | PPO | SAC |
|---|---|---|
| Policy type | Actor | Actor |
| Critic | 1 value critic typically | 2 Q critics |
| Learning | On-policy | Off-policy |
| Replay buffer | ❌ | ✅ |
| Experience reuse | Several epochs within rollout | Repeatedly across training |
| Policy update | Clipped objective | Q-guided policy optimization |
| Exploration | Policy stochasticity | Explicit entropy objective |
| Target critics | ❌ | ✅ |
| Sample efficiency | Usually lower | Usually higher |
| Continuous actions | ✅ | ✅ |
| Discrete actions | ✅ | Not standard SAC |

## 4. And the critic is fundamentally different

This is another important distinction.

### PPO

Critic learns:

"How much total future reward should I expect from this state?"

$$V(s)$$

Then PPO uses that to calculate advantages:

"Was this action better/worse than expected?"

That's why we spent so much time with:

$$\text{reward} \to \text{returns} \to \text{value} \to \text{GAE} \to \text{advantage}$$

### SAC

Critic learns:

"How valuable is taking THIS action in THIS state?"

$$Q(s,a)$$

And SAC has two:

$$Q_1(s,a)$$
$$Q_2(s,a)$$

Then the actor asks:

"Which actions do my critics think are valuable?"

```
state
  ↓
Actor
  ↓
action
  ↓
Q1/Q2
  ↓
how good?
  ↓
improve Actor
```

---

## 5. The exploration difference

### PPO:

```
policy distribution
       ↓
sample actions
       ↓
naturally explores
```

### SAC explicitly says:

```
good reward
     +
maintain useful entropy
     ↓
better policy
```

So SAC has an explicit mechanism encouraging exploration.

---

## 6. Why SAC can reuse old data

This is the architectural reason:

PPO's learning depends heavily on:

"How did my OLD policy behave?"

and uses the old/new policy relationship.

SAC instead learns Q-functions from transitions:

$$(s, a, r, s')$$

Those transitions can remain useful even after the actor changes.

### PPO

```
Policy₁ → data₁ → update
                     ↓
                  discard
```

### SAC

```
Policy₁ → data₁ ─────────┐
Policy₂ → data₂ ────────┤
Policy₃ → data₃ ────────┤
                         ↓
                    Replay Buffer
                         ↓
                    keep sampling
```

That's the core reason SAC can be much more sample-efficient.

# Level 1: Architecture

We're going to use one concrete example throughout:

A robot arm decides how much to move each joint.

So the RL policy is the decision-maker. We are not worrying about the downstream motor controller.

---

## 1. The complete SAC system

```
                         SAC
                          │
             ┌────────────┴────────────┐
             │                         │
          ACTOR                     CRITICS
             │                    ┌────┴────┐
             │                    │         │
             │                   Q1        Q2
             │
             ↓
        continuous action
             │
             ↓
        ENVIRONMENT
             │
       ┌─────┴─────┐
       ↓           ↓
    reward      next state
       │           │
       └─────┬─────┘
             ↓
       REPLAY BUFFER
             │
             ↓
        sample batch
             │
             ├──────────────→ Q1/Q2 training
             │
             └──────────────→ Actor training
```

There are also:

- Target Q1
- Target Q2

and:

- Temperature α

We'll get to both later.

---

## 2. Actor — the decision-maker

The actor receives the current state.

**Example:**

```
state =
[
    joint1_position,
    joint2_position,
    joint1_velocity,
    joint2_velocity
]
```

It passes that through a neural network:

```
state
  ↓
Actor NN
  ↓
action distribution
  ↓
sample
  ↓
action
```

For example:

```
action = [0.35, -0.72]
```

Meaning conceptually:

- joint 1 → move in this direction/amount
- joint 2 → move in this direction/amount

The important part:

**Actor outputs the RL action.**

---

## 3. Environment

The action goes into the environment:

```
state
  ↓
Actor
  ↓
action
  ↓
Environment
```

The environment responds:

- reward
- next_state
- done

**Example:**

```
action = [0.35, -0.72]

environment:
reward = +2.4
next_state = s₁
```

So we now have:

$$(s_0, a_0, r_0, s_1, \text{done})$$

This is one transition.

---

## 4. Replay Buffer

This entire transition goes into:

**Replay Buffer**

```
Replay Buffer
────────────────────────────
(s₀, a₀, r₀, s₁, done)
(s₁, a₁, r₁, s₂, done)
(s₂, a₂, r₂, s₃, done)
...
────────────────────────────
```

This is actual collected experience.

Nothing here is a prediction.

- state      ← environment
- action     ← actor
- reward     ← environment
- next_state ← environment

---

## 5. Q1 and Q2 — the critics

Now we have the critics.

Unlike PPO's typical critic:

$$V(\text{state})$$

SAC's critics evaluate a state + action pair:

$$Q(\text{state}, \text{action})$$

So:

```
state + action
      ↓
     Q1
      ↓
"how valuable is this?"
```

and separately:

```
state + action
      ↓
     Q2
      ↓
"how valuable is this?"
```

**Example:**

```
Q1(s₀,a₀) = 5.2
Q2(s₀,a₀) = 5.8
```

These are predictions, not actual rewards.

---

## 6. Why does SAC need Q instead of V?

Because SAC wants to answer:

"How good is this particular action?"

Imagine the same state:

$$s_0$$

Actor could produce:

- $a_1 = [0.2, 0.3]$
- $a_2 = [0.8,-0.7]$
- $a_3 = [-0.4,0.1]$

The Q critic can evaluate each:

```
Q(s₀,a₁) → 3.1
Q(s₀,a₂) → 8.7
Q(s₀,a₃) → 1.9
```

So SAC gets a direct signal about which actions are valuable.

That becomes extremely useful for training the actor later.

---

## 7. Target Q1 and Target Q2

Now we have another pair:

**Online critics:**
- Q1
- Q2

**Target critics:**
- Target Q1
- Target Q2

Think of them as:

```
Q1 ───────→ Target Q1
Q2 ───────→ Target Q2
       slow copy
```

The online critics are being trained.

The target critics provide a relatively stable reference for calculating future value.

They aren't another source of environment data.

We'll later walk through exactly how they calculate the target.

---

## 8. Temperature α

SAC also has:

$$\alpha$$

This controls how strongly SAC values exploration/entropy.

Conceptually:

```
Q value
   +
exploration pressure
   ↓
SAC learning objective
```

Don't worry about its mechanics yet.

For now just remember:

$$\alpha \text{ controls the tradeoff between getting high-value actions and maintaining exploration.}$$

---

## 9. Now classify EVERYTHING

This is important.

**Actual environment data**
- s
- a
- r
- s'
- done

Stored in replay buffer.

**Neural-network predictions**
- Q1(s,a)
- Q2(s,a)
- policy distribution

**Stable future-value machinery**
- Target Q1
- Target Q2

**Exploration control**
- α

---

## 10. One complete picture

```
                         ┌─────────────┐
                         │    ACTOR    │
                         │     NN      │
                         └──────┬──────┘
                                │
                             action
                                │
                                ↓
                         ┌─────────────┐
                         │ ENVIRONMENT │
                         └──────┬──────┘
                                │
                    ┌───────────┼───────────┐
                    ↓           ↓           ↓
                 reward      next state    done
                    │           │
                    └─────┬─────┘
                          ↓
                  ┌───────────────┐
                  │ REPLAY BUFFER │
                  └───────┬───────┘
                          │
                    sample batch
                          │
             ┌────────────┴────────────┐
             ↓                         ↓
       Target Q1/Q2                Q1/Q2
             │                         │
             │                    learn Q values
             │                         │
             └────────────┬────────────┘
                          ↓
                       ACTOR
                          ↓
                   improve policy

                  + α controls
                    exploration
```

---

## Key thing before Level 2

SAC does not work like:

```
rollout
→ calculate advantage
→ train
```

There is no PPO-style GAE/advantage pipeline here.

Instead:

```
experience
→ store transition
→ sample transition later
→ use reward + estimated future
→ create Q-learning target
→ train critics
→ use critics to train actor
```

# Level 2: Collecting Experience

Now we only collect data. No learning yet.

Our robot example:

```
state → Actor → action → Environment → reward + next_state
```

---

## 1. Start with state

Suppose:

$$s_0 = [0.2, 0.5]$$

Actor receives it:

```
s₀
 ↓
Actor NN
 ↓
action distribution
 ↓
sample
 ↓
a₀ = 0.7
```

The important distinction:

The actor doesn't necessarily output 0.7 deterministically. It outputs a distribution, and SAC samples from it.

---

## 2. Environment executes that action

```
s₀
 +
a₀ = 0.7
      ↓
Environment
```

Environment returns:

- reward = +2.0
- next_state = s₁
- done = False

So we now have one complete transition:

$$(s_0, a_0, r_0, s_1, \text{done})$$

Specifically:

```
([0.2, 0.5], 0.7, +2.0, [0.3, 0.6], False)
```

---

## 3. Store it

We put that exact data into the replay buffer:

**Replay Buffer**

```
T₀:
state      = [0.2, 0.5]
action     = 0.7
reward     = +2.0
next_state = [0.3, 0.6]
done       = False
```

Nothing has been calculated by the critics yet.

---

## 4. Next environment step

Current state is now:

$$s_1 = [0.3, 0.6]$$

Actor produces another action:

```
s₁
 ↓
Actor
 ↓
a₁ = 0.4
```

Environment:

```
a₁ = 0.4
 ↓
reward = +1.5
next_state = s₂
```

Store:

$$T_1 = (s_1, a_1, +1.5, s_2, \text{False})$$

---

## 5. Keep going

Eventually:

**Replay Buffer**

```
T₀ = (s₀, a₀, r₀, s₁, done)
T₁ = (s₁, a₁, r₁, s₂, done)
T₂ = (s₂, a₂, r₂, s₃, done)
T₃ = (s₃, a₃, r₃, s₄, done)
T₄ = (s₄, a₄, r₄, s₅, done)
...
```

After 10,000 interactions:

```
Replay Buffer
      ↓
10,000 transitions
```

---

## 6. STOP — what has actually happened?

At this point:

**Environment gave us**
- state
- action
- reward
- next_state
- done

**Actor did**
- state → action

**Critics did**
- Nothing yet.

### No advantage

Unlike PPO:

- ❌ no GAE
- ❌ no advantage calculation
- ❌ no rollout return calculation

### No policy ratio

- ❌ no old/new probability ratio

### No gradient update

- ❌ Actor not updated
- ❌ Q1 not updated
- ❌ Q2 not updated

We have simply collected and stored experience.

---

## 7. Now the important SAC difference

Suppose T₀ happened 1000 environment steps ago:

$$T_0 = (s_0, a_0, +2.0, s_1)$$

It is still sitting here:

```
Replay Buffer
      │
      ├── T₀ ← old
      ├── T₁
      ├── T₂
      ├── ...
      └── T₉₉₉₉ ← recent
```

Later, we can randomly sample:

$$T_0$$

again.

That is where SAC's old experience enters training.

Not through an advantage.

Not through a PPO ratio.

The raw transition itself is reused.

---

## Mental checkpoint

You should now have:

**COLLECTION PHASE**

```
Actor
 ↓
action
 ↓
Environment
 ↓
(s, a, reward, s')
 ↓
Replay Buffer
```

And the buffer contains raw historical experiences.

# Level 3: First Training Update

Now we finally train something.

We have a replay buffer:

```
T0 = (s0, a0, r0, s1, done)
T1 = (s1, a1, r1, s2, done)
T2 = (s2, a2, r2, s3, done)
...
```

Suppose we randomly sample:

$$T_0 = (s_0, a_0, +2.0, s_1, \text{False})$$

We are going to use this old transition to train the critics.

---

## Step 1 — What do we already know?

From the buffer:

```
s0       = [0.2, 0.5]
a0       = 0.7
reward   = +2.0
s1       = [0.3, 0.6]
done     = False
```

These are fixed historical facts.

We don't recalculate them.

---

## Step 2 — Ask the current Actor about the NEXT state

We take:

$$s_1$$

and feed it into the current actor:

```
s₁
 ↓
CURRENT ACTOR
 ↓
new action distribution
 ↓
sample
 ↓
a1_new
```

Suppose:

$$a_{1,\text{new}} = 0.4$$

Notice carefully:

The replay buffer had some action that was originally taken from s1:

$$T_1 = (s_1, a_{1,\text{old}}, ...)$$

But we're not using that old action here.

We're asking:

"With my policy NOW, what action would I take from s1?"

---

## Step 3 — Ask the Target Critics about that new action

We now have:

- $s_1$
- $a_{1,\text{new}} = 0.4$

Feed them into:

- Target Q1
- Target Q2

Maybe:

```
Target Q1(s1, 0.4) = 6.0
Target Q2(s1, 0.4) = 7.0
```

SAC takes the conservative one:

$$\min(6.0, 7.0) = 6.0$$

So we now have:

$$\text{future value} \approx 6.0$$

---

## Step 4 — Bring back the OLD reward

Here's the part you were asking about earlier.

Our replay transition contained:

$$\text{reward} = +2.0$$

We combine:

**REAL OLD REWARD**
+
**estimated future value**

Conceptually:

```
+2.0
 +
discounted 6.0
```

Suppose after discounting:

$$\text{target} = 7.4$$

That 7.4 is the target for the critics.

It means:

"Given what happened in this old transition, the estimated long-term value of (s0, a0) should be around 7.4."

---

## Step 5 — Now train Q1 and Q2

We feed the original stored state + original stored action into the current critics:

```
s0 + a0
   ↓
 Q1
 Q2
```

Suppose:

```
Q1(s0,a0) = 4.1
Q2(s0,a0) = 5.0
```

But our target is:

$$7.4$$

So the critics get trained toward:

```
Q1: 4.1 → 7.4
Q2: 5.0 → 7.4
```

Gradient descent changes their weights.

---

## Step 6 — What exactly got reused?

The old transition.

We used:

$$(s_0, a_0, +2.0, s_1)$$

from the replay buffer.

Specifically:

```
s0 + a0
   ↓
train current Q1/Q2

reward + s1
   ↓
construct their training target
```

So the old data isn't merely sitting there.

Every sampled transition actively produces a gradient update.

---

## Step 7 — And why can we use it again?

Suppose 50 updates later:

```
Replay Buffer
      ↓
random sample
      ↓
T0 gets selected again
```

We again retrieve:

$$(s_0, a_0, +2.0, s_1)$$

But now our networks have changed.

So we might get:

**old update:**

```
future estimate = 6.0
target = 7.4
```

**later update:**

```
future estimate = 8.2
target = 9.1
```

Same historical transition.

Same stored reward.

Different current network predictions.

That's the mechanism that makes the old experience useful repeatedly.

---

## One crucial distinction

Notice what SAC doesn't do:

- ❌ calculate GAE
- ❌ calculate advantage
- ❌ calculate old/new policy ratio
- ❌ protect the transition with clipping

Instead:

```
Replay transition
(s,a,r,s')
       │
       ├── s,a ─────────→ current Q1/Q2
       │
       └── r,s'
             │
             ↓
       current actor
             ↓
         new action
             ↓
       target Q1/Q2
             ↓
       future estimate
             │
             ↓
     reward + future
             │
             ↓
         Q target
             │
             ↓
       train Q1/Q2
```

# Level 4: Critics and Training

## 1. Two Critics — Q1 and Q2

SAC uses:

$$Q_1(s,a)$$
$$Q_2(s,a)$$

When estimating future value:

$$\min(Q_1, Q_2)$$

### Why?

Neural critics tend to overestimate Q-values.

If one says:

```
Q1 = 10
Q2 = 7
```

SAC uses:

$$7$$

This reduces optimistic/unstable learning.

This is called **Clipped Double Q-learning**.

---

## 2. Target Critics

There are also:

$$Q_{1,\text{target}}$$
$$Q_{2,\text{target}}$$

They are slowly updated copies of the online critics.

```
Online Q
   │
   └── slowly → Target Q
```

They are used to calculate the future-value target.

### Why?

Without them, the network would be trying to learn from a target that is moving every time the network itself changes.

Target networks make the Bellman target more stable.

---

## 3. The SAC Target

For a sampled transition:

$$(s, a, r, s', \text{done})$$

SAC:

```
s'
 ↓
current Actor
 ↓
new action a'
 ↓
Target Q1(s',a')
Target Q2(s',a')
 ↓
min()
 ↓
future value
```

Then combines:

- real reward
- discounted future value
- entropy term

to create the critic target.

### PPO vs SAC

**PPO:**
```
reward sequence → return/GAE → advantage
```

**SAC:**
```
reward + next-state Q estimate → Bellman target
```

That's one of the biggest conceptual differences.

---

## 4. Entropy — SAC's signature idea

SAC doesn't want:

"Find the highest-reward action immediately."

It wants:

"Find good actions while keeping the policy sufficiently exploratory."

So its objective contains:

$$Q \text{ value} + \text{entropy}$$

**High entropy** means the policy remains more spread out.

**Low entropy** means it becomes more deterministic.

This prevents premature collapse into one possibly-bad strategy.

---

## 5. Temperature α

$$\alpha$$  controls how much entropy matters.

```
high α
→ exploration matters more

low α
→ reward/Q matters more
```

Modern SAC usually learns α automatically rather than forcing you to choose it perfectly.

So training has another optimization:

- Actor weights
- Critic weights
- α

all being updated.

---

## 6. Reparameterization Trick

This one matters for implementation.

The Actor samples:

$$a \sim \pi(a|s)$$

But we need gradients to flow:

```
Q(s,a)
   ↓
Actor
```

So SAC uses a differentiable transformation of noise:

```
random noise ε
      ↓
Actor → μ, σ
      ↓
sample z
      ↓
tanh(z)
      ↓
action
```

Conceptually:

```
randomness + actor parameters
            ↓
      differentiable action
```

Therefore the gradient can travel:

```
Q
 ↓
action
 ↓
Actor
 ↓
Actor weights
```

This is a **critical implementation detail**.

---

## 7. Why tanh?

The Gaussian policy can produce unbounded values.

But environments usually have action limits:

$$[-1, +1]$$

So SAC commonly does:

```
Gaussian sample
      ↓
     tanh
      ↓
[-1, +1] action
```

If the environment's actual range is:

$$[-2, +2]$$

you scale the result accordingly.

There's also an important log-probability correction because tanh transforms the distribution.

You need this correction in a real implementation.

---

## 8. Replay Buffer Sampling

SAC doesn't process the buffer sequentially.

Suppose:

$$1,000,000 \text{ transitions}$$

It randomly samples:

$$256 \text{ transitions}$$

and trains on that mini-batch.

Then another random batch.

Then another.

Old and new experiences can therefore appear together:

```
old old new old new new old ...
```

This breaks strong temporal correlation and improves data efficiency.

---

## 9. One SAC Training Iteration

This is the whole thing:

```
                 ENVIRONMENT
                     ↑
                     │
                   action
                     │
                   ACTOR
                     │
                     ↓
              Replay Buffer
                     │
                random batch
                     │
          ┌──────────┴──────────┐
          │                     │
          ↓                     ↓
     CRITIC UPDATE         ACTOR UPDATE
          │                     │
   s,a,r,s'                  s
          │                     │
          ↓                  Actor
    next action                ↓
          ↓               fresh action
   Target Q1/Q2                ↓
          ↓                 Q1/Q2
       min Q                   ↓
          ↓              Q + entropy
   reward + future              ↓
          ↓                 Actor loss
     Q target                    ↓
          ↓                 Actor update
    Q1/Q2 loss
          ↓
    Q1/Q2 update

              ↓
         update α

              ↓
      soft-update targets

Then repeat.
```

---

## 10. Soft Target Update

Target critics aren't hard-copied every iteration.

Instead:

$$\text{target} = \tau \times \text{online} + (1-\tau) \times \text{target}$$

with small τ, e.g. 0.005.

Meaning:

```
Online Q changes quickly
        ↓
Target Q follows slowly
```

This stabilizes training.

---

## 11. What gets updated vs what doesn't

### During critic update
✅ Q1 weights
✅ Q2 weights

❌ Actor
❌ Target Q directly

### During actor update
✅ Actor weights

❌ Q1/Q2 weights

Critics are used as evaluators here.

### During α update
✅ α

### Target update
- Target Q1 ← slowly follow Q1
- Target Q2 ← slowly follow Q2

---

## 12. The entire data lifecycle

This is the thing I want you to remember:

```
                    COLLECT
                       ↓
                (s,a,r,s',done)
                       ↓
                 STORE FOREVER*
                       ↓
                RANDOM SAMPLE
                       ↓
             ┌─────────┴─────────┐
             ↓                   ↓
        CRITIC TARGET        CURRENT ACTOR
             ↓                   ↓
        Q1/Q2 UPDATE         fresh action
                                 ↓
                            Q evaluation
                                 ↓
                            ACTOR UPDATE
                                 ↓
                              α UPDATE
                                 ↓
                         TARGET Q SOFT UPDATE
                                 ↓
                              REPEAT
```

\* Until the replay buffer fills up; then old experiences are evicted.